# 🧠 Cerveau de FactuBot — entraînement puissant (gratuit sur Kaggle)

FactuBot apprend le français **de zéro** : un mini-GPT d'environ 14 millions de neurones lit des centaines de milliers de phrases libres de droits (Wikipédia en français + livres du domaine public de la BnF / Gallica). Ensuite, un **juge strict** (LanguageTool) corrige ses phrases, et seules les phrases parfaites servent à le ré-entraîner.

### À faire une seule fois (5 minutes)
1. Crée un compte gratuit sur **kaggle.com** et vérifie ton numéro de téléphone : c'est nécessaire pour avoir Internet et une carte graphique.
2. **Create → New Notebook**, puis **File → Import Notebook** et choisis ce fichier.
3. Dans le panneau de droite, **Settings** :
   - **Accelerator** : `GPU T4 x2` (ou `P100`)
   - **Internet** : `On`
4. Clique sur **Run All**. Compte environ **3 h 30**. Tu peux fermer l'onglet si tu utilises **Save Version → Save & Run All (Commit)**.
5. À la fin, dans l'onglet **Output**, télécharge **`cerveau.bin`** et envoie-le à Claude : il le branchera sur le site.


In [ ]:
# 1. outils
!pip install -q tokenizers pyarrow huggingface_hub language_tool_python
!apt-get -qq install -y openjdk-17-jre-headless > /dev/null 2>&1 || echo "Java non installe : le juge passera par l'API publique"
import torch; print('GPU :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'AUCUN (active le GPU dans Settings !)')

In [ ]:
# 2. le code de FactuBot (depot public)
!git clone -q --depth 1 -b claude/practical-knuth-c3vpo2 https://github.com/zelkoappstore-cloud/robloxprojecy.git fb
!ls fb/factubot/cerveau

In [ ]:
# 3. les lectures : Wikipedia FR (CC BY-SA) + livres du domaine public (Gallica / PleIAs)
from huggingface_hub import hf_hub_download
files = []
for i in [1, 2, 5, 9]:
    files.append(hf_hub_download('wikimedia/wikipedia', f'20231101.fr/train-{i:05d}-of-00017.parquet', repo_type='dataset', local_dir='data'))
for i in [1, 2, 3]:
    files.append(hf_hub_download('PleIAs/French-PD-Books', f'gallica_mono_{i}.parquet', repo_type='dataset', local_dir='data'))
print(files)

In [ ]:
# 4. nettoyage : une phrase propre par ligne (environ 400 Mo de francais)
import subprocess
!python fb/factubot/cerveau/prepare.py train.txt 400 {' '.join(files)}

In [ ]:
# 5. entrainement : lecture (environ 2 h 15) puis 4 tours de correction par le juge strict (environ 45 min)
import shutil
juge = 'local' if shutil.which('java') else 'api'
!python fb/factubot/cerveau/train.py --corpus train.txt --out /kaggle/working/cerveau.bin --taille gpu --minutes 180 --juge {juge} --tours 4 --phrases-par-tour 1000

### ✅ Terminé
Télécharge **`cerveau.bin`** dans l'onglet **Output** (à droite) et envoie-le à Claude dans la conversation. Il remplacera le petit cerveau actuel du site par celui-ci, beaucoup plus fort.

*Sources : Wikipédia en français (licence CC BY-SA 4.0) et corpus PleIAs French-PD-Books (livres du domaine public numérisés par la BnF / Gallica).*